# RAY-IMAGE v0.2 — Token-Conditioned 12-Class Learning Test

This notebook trains the RAY-IMAGE prototype with a frozen VAE, deterministic VAE mean latents, and a DiT with token-level text cross-attention. The architecture changes are inspired by successful text-to-image DiT systems such as PixArt-α while keeping the implementation small enough for a free T4 experiment.

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Select a GPU runtime and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!pip install -q -r requirements.txt

In [ ]:
!python -m ray_image.train_smoke

In [ ]:
!rm -rf data/toy
!python tools/make_toy_dataset.py --output data/toy --samples 2048 --size 64 --seed 1337

In [ ]:
!python -m ray_image.train_vae --manifest data/toy/manifest.jsonl --steps 1200 --batch-size 32 --save /content/ray_vae_v0_2.pt

## N1 — VAE latent sanity + latent statistics (DIAGNOSTIC ONLY)

This section does **not** train anything and does **not** change the architecture.
It answers: *"Does the current VAE latent preserve color AND shape well enough for
the generator to learn them?"* using the VAE trained in the previous cell.

What it does (all read-only w.r.t. weights; VAE runs in eval mode):
1. Draws one deterministic 64x64 reference image per toy class.
2. Encodes each with the **deterministic mean latent** (no sampling noise) and
   decodes it back to a reconstruction PNG (saved to `/content/n1/reconstructions/`).
3. Runs the **existing** 12-class evaluator on those reconstructions and reports
   color / shape / suite accuracy.
4. Estimates latent statistics over a dataset sample and writes
   `/content/n1/vae_latent_stats.json` (reused by N2 later).

**Read the numbers, then STOP before the long generator run below.**
Interpretation guide:
- If the *reconstruction* evaluator already loses shape (shape_accuracy is low
  even on clean decoded images) -> the VAE itself is a bottleneck (N1 conclusion).
- If color/shape are both high on reconstructions -> the VAE is fine, and the
  generator's collapse (from the earlier experiment) is a generator/conditioning
  problem, motivating N2/N4.
- Note the per-channel mean/std in the stats JSON: latents far from zero-mean /
  unit-variance hint that N2 latent whitening would help balance the flow loss.


In [ ]:
# N1 — run the VAE latent probe (diagnostic; no training)
!mkdir -p /content/n1
!python -m ray_image.probe_vae_latents \
  --vae-checkpoint /content/ray_vae_v0_2.pt \
  --manifest data/toy/manifest.jsonl \
  --outdir /content/n1 \
  --size 64 --seed 1337 \
  --stats-samples 512 --stats-batch 32
print('\nArtifacts:'); !ls -1 /content/n1/reconstructions | head


In [ ]:
!python -m ray_image.train_generator --manifest data/toy/manifest.jsonl --vae /content/ray_vae_v0_2.pt --steps 8000 --batch-size 16 --save /content/ray_image_v0_2_trained.pt

In [ ]:
from pathlib import Path
prompts = [
    ('red_circle', 'a red circle'), ('red_square', 'a red square'), ('red_triangle', 'a red triangle'),
    ('green_circle', 'a green circle'), ('green_square', 'a green square'), ('green_triangle', 'a green triangle'),
    ('blue_circle', 'a blue circle'), ('blue_square', 'a blue square'), ('blue_triangle', 'a blue triangle'),
    ('yellow_circle', 'a yellow circle'), ('yellow_square', 'a yellow square'), ('yellow_triangle', 'a yellow triangle'),
]
for name, prompt in prompts:
    out = Path('/content/ray_suite') / f'{name}.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    !python -m ray_image.generate --checkpoint /content/ray_image_v0_2_trained.pt --prompt "{prompt}" --steps 50 --seed 42 --output "{out}"
    print(name, 'exists=', out.exists(), 'bytes=', out.stat().st_size if out.exists() else '-')

In [ ]:
!python tools/evaluate_toy_suite.py --dir /content/ray_suite

## Target

First target: beat the previous 1/12 (8.3%) semantic suite score and stop collapsing to a single class. A strong milestone is >75% on the fixed 12-class toy task. Only after that do we move to larger compositions and real anime data.